In [1]:
from transformers.pipelines import SUPPORTED_TASKS
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch


# for i, v in SUPPORTED_TASKS.items():
#     print(i, v)

g:\software\anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# pipe = pipeline("text-classification", model="uer/roberta-base-finetuned-dianping-chinese")

Device set to use cuda:0


In [2]:
tokenizer = AutoTokenizer.from_pretrained("G:/model_weights/huggingface_model/roberta-base-finetuned-dianping-chinese")
model = AutoModelForSequenceClassification.from_pretrained("G:/model_weights/huggingface_model/roberta-base-finetuned-dianping-chinese")
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
pipe("垃圾")
# model.save_pretrained("G:/model_weights/huggingface_model/roberta-base-finetuned-dianping-chinese")
# tokenizer.save_pretrained("G:/model_weights/huggingface_model/roberta-base-finetuned-dianping-chinese")
model.config

Device set to use cuda:0


BertConfig {
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative (stars 1, 2 and 3)",
    "1": "positive (stars 4 and 5)"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "negative (stars 1, 2 and 3)": 0,
    "positive (stars 4 and 5)": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 21128
}

In [21]:
import time
times = []
for i in range(100):
    torch.cuda.synchronize()
    start = time.time()
    pipe("我觉得不太行！")
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)
print(f"平均推理时间: {sum(times)/len(times)} 秒")

平均推理时间: 0.005164353847503662 秒


In [43]:
inputs = "我觉得这个产品非常好！"
inputs = tokenizer(inputs, return_tensors="pt")
inputs.to("cuda")

{'input_ids': tensor([[ 101, 2769, 6230, 2533, 6821,  702,  772, 1501, 7478, 2382, 1962, 8013,
          102]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [45]:
res = model(**inputs)

In [46]:
res.logits

tensor([[-1.3238,  1.3311]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [61]:
predit = torch.softmax(res.logits, dim=-1).argmax().item()
predit

1

In [62]:
label = model.config.id2label.get(predit)
label

'positive (stars 4 and 5)'

In [7]:
tokens = tokenizer.tokenize("顺义区数据和智慧城市底座建设项目")

In [ ]:
tokenizer.vocab

In [22]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids = tokenizer.encode("顺义区数据和智慧城市底座建设项目", add_special_tokens=True)
ids

[101,
 7556,
 721,
 1277,
 3144,
 2945,
 1469,
 3255,
 2716,
 1814,
 2356,
 2419,
 2429,
 2456,
 6392,
 7555,
 4680,
 102]

In [23]:
tokenizer.convert_tokens_to_string(tokens)

'顺 义 区 数 据 和 智 慧 城 市 底 座 建 设 项 目'

In [24]:
tokenizer.decode(ids, skip_special_tokens=False)  # 和上面等价

'[CLS] 顺 义 区 数 据 和 智 慧 城 市 底 座 建 设 项 目 [SEP]'

In [26]:
tokenizer.encode("顺义区数据和智慧城市底座建设项目", add_special_tokens=True, padding="max_length", max_length=20)

[101,
 7556,
 721,
 1277,
 3144,
 2945,
 1469,
 3255,
 2716,
 1814,
 2356,
 2419,
 2429,
 2456,
 6392,
 7555,
 4680,
 102,
 0,
 0]

In [ ]:
tokenizer.encode_plus("顺义区数据和智慧城市底座建设项目", padding="max_length", max_length=20)